In [2]:
import sys
sys.path.append('..')

from utils.spark_session import get_spark_session
from utils.parquet_io import read_parquet, write_partitioned_parquet
from pyspark.sql.functions import col, dayofweek, hour, when, round, broadcast

# Initialize Spark session
spark = get_spark_session()
print('session created')

session created


In [3]:
# Read parquet files
taxi01_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\03_feature_data\\taxi01")
taxi02_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\03_feature_data\\taxi02")
zone_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\03_feature_data\\zone")

In [6]:
zone_cache = zone_df.cache()

In [7]:
# Broadcast join

taxi01_zone_df = taxi01_df.join(
    broadcast(zone_cache),
    taxi01_df.PULocationID == zone_cache.LocationID,
    "left"
)

taxi02_zone_df = taxi02_df.join(
    broadcast(zone_cache),
    taxi02_df.PULocationID == zone_cache.LocationID,
    "left"
)


In [16]:
taxi02_zone_df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------------------+------------------+----------+------------+----------+---------+--------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|trip_duration_minutes|average_speed_kmph|is_weekend|is_rush_hour|LocationID|  Borough|                Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+----------------

In [ ]:
taxi01_zone_df.groupBy("VendorID").count().show()

PySparkRuntimeError: [SESSION_OR_CONTEXT_NOT_EXISTS] SparkContext or SparkSession should be created first.

In [21]:
# Save join data

write_partitioned_parquet(taxi01_zone_df, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\07_optimized_data\\taxi01", "VendorID")

In [26]:
write_partitioned_parquet(taxi02_zone_df, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\07_optimized_data\\taxi02", "is_weekend")

In [27]:
print("Data saved successfully!")

spark.stop()

Data saved successfully!
